# AI 2002 – Assignment 2: UNO Game AI
### Adversarial Search: Minimax (Defensive) vs Expectimax (Offensive)
**GitHub Repository:** https://github.com/AhmedIftikhar-datascience/UNO-GAME-AI

## 1. Card Class & Deck Generator

In [2]:
import random
import copy

# Creates a basic UNO card with a color and a value
class Card:
    def __init__(self, color, value):
        self.color = color   # Colors like Red, Blue, Green, Yellow
        self.value = value   # Numbers 0-9 or 'Skip'

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        return self.color == other.color and self.value == other.value


# Makes a full deck of cards and mixes them up
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []

    for color in colors:
        for number in range(10):          # Add number cards (0 to 9)
            deck.append(Card(color, number))
        deck.append(Card(color, 'Skip'))  # Add one Skip card per color

    random.shuffle(deck)  # Shuffle the deck
    return deck


# Gives 5 cards to each player
def deal_cards(deck, num_players=3, cards_each=5):
    hands = [[] for _ in range(num_players)]
    for _ in range(cards_each):
        for i in range(num_players):
            hands[i].append(deck.pop())
    return hands


print("Card class and deck generator ready.")
print("Sample card:", Card('Red', 5))
print("Sample Skip:", Card('Blue', 'Skip'))

Card class and deck generator ready.
Sample card: Red 5
Sample Skip: Blue Skip


## 2. Legal Move Generator & State Transition

In [3]:
# Finds cards that can be played on the current top card
def get_valid_moves(hand, top_card):
    valid = []
    for card in hand:
        # Match by color or value
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid


# Stores the current status of the game in a dictionary
def create_state(hands, top_card, deck):
    return {
        'p1_cards': hands[0],   # Player 1
        'p2_cards': hands[1],   # Player 2
        'p3_cards': hands[2],   # Player 3
        'top_card': top_card,
        'deck': deck,
        'current_player': 0,    # 0=P1, 1=P2, 2=P3
        'skip_next': False      # True if the next turn is skipped
    }


# Updates the game state after a player plays a card or draws
def apply_move(state, move, player_index):
    new_state = copy.deepcopy(state)  # Create a copy so we don't change the original state
    hand_key = ['p1_cards', 'p2_cards', 'p3_cards'][player_index]

    if move is None:
        # Draw a card (create a new deck if empty)
        if not new_state['deck']:
            new_state['deck'] = generate_deck()
        drawn = new_state['deck'].pop()
        new_state[hand_key].append(drawn)
    else:
        # Remove the played card from the hand
        new_state[hand_key] = [c for c in new_state[hand_key]
                                if not (c.color == move.color and c.value == move.value)]
        new_state['top_card'] = move

        # Handle Skip cards
        if move.value == 'Skip':
            new_state['skip_next'] = True
        else:
            new_state['skip_next'] = False

    # Move to the next player's turn
    new_state['current_player'] = (player_index + 1) % 3
    return new_state


# Checks if the game is over (someone has 0 cards)
def is_terminal(state):
    return (len(state['p1_cards']) == 0 or
            len(state['p2_cards']) == 0 or
            len(state['p3_cards']) == 0)


print("Legal move generator and state transition ready.")

Legal move generator and state transition ready.


## 3. Evaluation Function

### Formula:
Score = 50 - 5(C_{AI}) + 2(C_{opp}) + 3(S)$$

- **C_AI**: Cards in the current player's hand (fewer = better, so subtracted)
- **C_opp**: Average cards held by the two opponents (more = better for us)
- **S**: Number of Skip cards in hand (powerful for disruption)

**Weight tuning:**
- **Defensive (Minimax)**: Higher penalty on own cards (`-5`), emphasizes Skip cards (`+3`) to block opponents
- **Offensive (Expectimax)**: Higher reward on opponent cards (`+2`), encourages fast card-shedding

In [4]:
# Scores the current game based on the AI's play style (defensive or offensive)
def evaluate(state, player_index, strategy='defensive'):
    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    my_hand = state[keys[player_index]]
    opp_hands = [state[keys[i]] for i in range(3) if i != player_index]

    C_AI  = len(my_hand)                                    # How many cards I have
    C_opp = sum(len(h) for h in opp_hands) / len(opp_hands) # Average cards opponents have
    S     = sum(1 for c in my_hand if c.value == 'Skip')    # How many Skip cards I have

    if strategy == 'defensive':
        # Defensive style: hate having cards, love having Skips
        w_ai, w_opp, w_skip = 5, 2, 3
    else:
        # Offensive style: care more about opponents having lots of cards
        w_ai, w_opp, w_skip = 4, 3, 2

    score = 50 - w_ai * C_AI + w_opp * C_opp + w_skip * S
    return round(score, 2)


print("Evaluation function ready.")
print("Formula: Score = 50 - w_ai*C_AI + w_opp*C_opp + w_skip*S")
print("Defensive weights: w_ai=5, w_opp=2, w_skip=3")
print("Offensive weights: w_ai=4, w_opp=3, w_skip=2")

Evaluation function ready.
Formula: Score = 50 - w_ai*C_AI + w_opp*C_opp + w_skip*S
Defensive weights: w_ai=5, w_opp=2, w_skip=3
Offensive weights: w_ai=4, w_opp=3, w_skip=2


## 4. Minimax Search (Player 1 – Defensive)

In [5]:
# Player 1 uses this defensive strategy.
# It plans ahead by assuming opponents will always try to give Player 1 the lowest score.
def minimax(state, depth, maximizing, player_index, root_player):
    # Stop searching if the game is over or we've looked far enough ahead
    if depth == 0 or is_terminal(state):
        return evaluate(state, root_player, strategy='defensive'), None

    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    hand = state[keys[player_index]]
    valid_moves = get_valid_moves(hand, state['top_card'])

    # Add "draw a card" (represented by None) to the list of possible moves
    actions = valid_moves if valid_moves else []
    actions = actions + [None]  

    next_player = (player_index + 1) % 3

    if maximizing:
        # Try to get the highest possible score for our player
        best_score = float('-inf')
        best_move = None
        for move in actions:
            new_state = apply_move(state, move, player_index)
            
            # If a Skip card is played, skip the next player's turn
            if new_state['skip_next']:
                next_p = (next_player + 1) % 3
                new_state['skip_next'] = False
            else:
                next_p = next_player
                
            score, _ = minimax(new_state, depth - 1, False, next_p, root_player)
            if score > best_score:
                best_score = score
                best_move = move
        return best_score, best_move
    else:
        # Opponents try to give our player the lowest possible score
        best_score = float('inf')
        best_move = None
        for move in actions:
            new_state = apply_move(state, move, player_index)
            
            # If a Skip card is played, skip the next player's turn
            if new_state['skip_next']:
                next_p = (next_player + 1) % 3
                new_state['skip_next'] = False
            else:
                next_p = next_player
                
            # Check if it's our main player's turn next so we can go back to maximizing
            is_max = (next_p == root_player)
            score, _ = minimax(new_state, depth - 1, is_max, next_p, root_player)
            if score < best_score:
                best_score = score
                best_move = move
        return best_score, best_move


print("Minimax (Defensive) search ready. Depth = 3")

Minimax (Defensive) search ready. Depth = 3


## 5. Expectimax Search (Player 2 – Offensive)

In [6]:
# Player 2 uses this offensive strategy.
# It plans ahead by finding the best move, calculating the odds of drawing 
# helpful cards, and assuming opponents will just play a random legal card.
def expectimax(state, depth, player_index, root_player):
    # Stop searching if the game is over or we've looked far enough ahead
    if depth == 0 or is_terminal(state):
        return evaluate(state, root_player, strategy='offensive'), None

    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    hand = state[keys[player_index]]
    valid_moves = get_valid_moves(hand, state['top_card'])
    next_player = (player_index + 1) % 3

    if player_index == root_player:
        # Our turn: test all valid moves to find the one with the highest score
        best_score = float('-inf')
        best_move = None

        for move in valid_moves:
            new_state = apply_move(state, move, player_index)
            next_p = (next_player + 1) % 3 if new_state['skip_next'] else next_player
            new_state['skip_next'] = False
            score, _ = expectimax(new_state, depth - 1, next_p, root_player)
            if score > best_score:
                best_score = score
                best_move = move

        # Consider the option of drawing a card
        # We calculate the likely score based on what cards are left in the deck
        if state['deck']:
            deck = state['deck']
            total = len(deck)
            expected_score = 0.0

            # Group the remaining cards to figure out the odds of drawing each type
            seen = {}
            for c in deck:
                key = (c.color, c.value)
                seen[key] = seen.get(key, 0) + 1

            for (color, value), count in seen.items():
                prob = count / total
                
                # Pretend we drew this specific card to see what happens
                sim_state = copy.deepcopy(state)
                drawn = Card(color, value)
                sim_state[keys[player_index]].append(drawn)
                sim_state['deck'] = [c for c in sim_state['deck']
                                     if not (c.color == color and c.value == value)]
                if sim_state['deck']:  
                    pass
                sim_state['current_player'] = next_player
                score, _ = expectimax(sim_state, depth - 1, next_player, root_player)
                expected_score += prob * score

            expected_score = round(expected_score, 2)

            if expected_score > best_score:
                best_score = expected_score
                best_move = None  # None means we choose to draw

        # If we have no valid moves and the deck is empty, we are forced to pass
        if not valid_moves and not state['deck']:
            return evaluate(state, root_player, 'offensive'), None

        return best_score, best_move

    else:
        # Opponent's turn: assume they just play a random legal card
        if valid_moves:
            move = random.choice(valid_moves)
        else:
            move = None  # Draw a card
            
        new_state = apply_move(state, move, player_index)
        next_p = (next_player + 1) % 3 if new_state['skip_next'] else next_player
        new_state['skip_next'] = False
        score, _ = expectimax(new_state, depth - 1, next_p, root_player)
        return score, move


print("Expectimax (Offensive) search ready. Depth = 3")
print("Nodes: MAX (AI turn) | CHANCE (draw) | OPPONENT (random legal move)")

Expectimax (Offensive) search ready. Depth = 3
Nodes: MAX (AI turn) | CHANCE (draw) | OPPONENT (random legal move)


## 6. Game Tree Printer
Generates and prints the search tree for a sample state (for deliverable requirement)

In [7]:
# Draws a visual map (a tree) of the possible future moves.
# This helps us see how the AI is thinking ahead.
tree_lines = []  # Saves the lines of text to print the tree later

def print_tree(state, depth, player_index, root_player, prefix='', algorithm='minimax', max_depth=2):
    if depth == 0 or is_terminal(state):
        score = evaluate(state, root_player,
                        'defensive' if algorithm == 'minimax' else 'offensive')
        line = prefix + f"[LEAF] Score={score}"
        tree_lines.append(line)
        print(line)
        return

    keys = ['p1_cards', 'p2_cards', 'p3_cards']
    player_name = ['P1(Minimax)', 'P2(Expectimax)', 'P3'][player_index]
    hand = state[keys[player_index]]
    valid_moves = get_valid_moves(hand, state['top_card'])
    actions = valid_moves[:2] + [None]  # Only show up to 2 card plays plus drawing so the tree isn't too huge

    node_type = 'MAX' if player_index == root_player else 'MIN/OPP'
    if player_index == root_player and algorithm == 'expectimax':
        node_type = 'MAX'

    line = prefix + f"[{node_type}] {player_name} | Top: {state['top_card']} | Hand size: {len(hand)}"
    tree_lines.append(line)
    print(line)

    for i, action in enumerate(actions):
        connector = '└── ' if i == len(actions) - 1 else '├── '
        child_prefix = prefix + ('    ' if i == len(actions) - 1 else '│   ')
        action_label = str(action) if action else 'DRAW'

        # Show the odds if the AI decides to draw a card
        if action is None and algorithm == 'expectimax' and player_index == root_player:
            action_label = f'CHANCE (Draw, P={round(1/max(len(state["deck"]),1),2)})'

        move_line = prefix + connector + f"Action: {action_label}"
        tree_lines.append(move_line)
        print(move_line)

        new_state = apply_move(state, action, player_index)
        next_player = (player_index + 1) % 3
        print_tree(new_state, depth - 1, next_player, root_player,
                   child_prefix, algorithm, max_depth)


print("Game tree printer ready.")

Game tree printer ready.


## 7. Player 3: Manual Mode & AI Simulation Mode

In [8]:
# Handles Player 3's turn. 
# It can be played by a real person ('manual') or by the computer ('simulation').
def player3_move(state, mode='simulation'):
    hand = state['p3_cards']
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if mode == 'manual':
        print("\n🎴 Your hand (Player 3):")
        for i, card in enumerate(hand):
            print(f"  [{i}] {card}")
        print(f"Top card: {top_card}")
        print("Valid moves:", valid_moves if valid_moves else "None – you must draw")

        if not valid_moves:
            input("Press Enter to draw a card...")
            return None  # Draw a card

        while True:
            try:
                choice = int(input("Enter index of card to play (or -1 to draw): "))
                if choice == -1:
                    return None
                if 0 <= choice < len(hand) and hand[choice] in valid_moves:
                    return hand[choice]
                else:
                    print("Invalid choice. Try again.")
            except ValueError:
                print("Please enter a number.")

    else:  # If simulation mode, let the computer play using the defensive strategy
        score, best_move = minimax(state, depth=3, maximizing=True,
                                   player_index=2, root_player=2)
        return best_move


print("Player 3 move handler ready (manual + simulation modes).")

Player 3 move handler ready (manual + simulation modes).


## 8. Main Game Loop

In [11]:
# Runs the full UNO game from start to finish.
# p3_mode determines if Player 3 is a real person ('manual') or the computer ('simulation').
def play_game(p3_mode='simulation'):
    print("=" * 55)
    print("         UNO GAME AI – Adversarial Search ")
    print("=" * 55)
    print(f"Mode: P1=Minimax(Defensive), P2=Expectimax(Offensive), P3={'User' if p3_mode=='manual' else 'AI-Minimax'}")
    print()

    # Create the deck and deal 5 cards to each player
    deck = generate_deck()
    hands = deal_cards(deck, num_players=3, cards_each=5)

    # Flip the first card to start the game (make sure it isn't a Skip card)
    top_card = deck.pop()
    while top_card.value == 'Skip':
        deck.insert(0, top_card)
        top_card = deck.pop()

    state = create_state(hands, top_card, deck)

    print(f"Starting top card: {top_card}")
    print(f"P1 hand: {hands[0]}")
    print(f"P2 hand: {hands[1]}")
    print(f"P3 hand: {hands[2]}")
    print(f"Deck size: {len(deck)} cards remaining")
    print()

    turn_count = 0
    max_turns = 100  # Stop the game if it goes on for too long

    # Keep playing until someone runs out of cards or we hit the turn limit
    while not is_terminal(state) and turn_count < max_turns:
        current = state['current_player']
        player_name = ['Player 1 (Minimax)', 'Player 2 (Expectimax)', 'Player 3'][current]
        keys = ['p1_cards', 'p2_cards', 'p3_cards']
        hand = state[keys[current]]
        top = state['top_card']

        print(f"── Turn {turn_count + 1}: {player_name} ──")
        print(f"   Top card : {top}")
        print(f"   Hand     : {hand}")

        # If the previous player used a Skip card, skip this turn
        if state['skip_next']:
            print(f"     {player_name} is SKIPPED!")
            state['skip_next'] = False
            state['current_player'] = (current + 1) % 3
            turn_count += 1
            print()
            continue

        valid = get_valid_moves(hand, top)
        print(f"   Valid moves: {valid if valid else ['Draw']}")

        # Figure out what move to make based on whose turn it is
        if current == 0:
            # Player 1 uses the defensive strategy
            score, move = minimax(state, depth=3, maximizing=True,
                                  player_index=0, root_player=0)
            action_label = str(move) if move else "Draw"
            print(f"    P1 Minimax decision: {action_label} (score={score})")

        elif current == 1:
            # Player 2 uses the offensive strategy
            score, move = expectimax(state, depth=3, player_index=1, root_player=1)
            action_label = str(move) if move else "Draw"
            print(f"    P2 Expectimax decision: {action_label} (score={score})")

            # Show the math behind Player 2's choices
            print(f"   All moves considered (depth=1 preview):")
            for m in (valid + [None]):
                tmp_state = apply_move(state, m, 1)
                s = evaluate(tmp_state, 1, 'offensive')
                print(f"     → {str(m) if m else 'Draw'}: Expected score = {s}")

        else:
            # Player 3 is either the user or the computer
            move = player3_move(state, mode=p3_mode)
            action_label = str(move) if move else "Draw"
            print(f"    P3 plays: {action_label}")

        # Update the game with the chosen move
        state = apply_move(state, move, current)

        print(f"   Cards remaining → P1:{len(state['p1_cards'])} | P2:{len(state['p2_cards'])} | P3:{len(state['p3_cards'])}")
        print()
        turn_count += 1

    # Print the final results
    print("=" * 55)
    print("  GAME OVER")
    if len(state['p1_cards']) == 0:
        print(" Player 1 (Minimax – Defensive) WINS!")
    elif len(state['p2_cards']) == 0:
        print(" Player 2 (Expectimax – Offensive) WINS!")
    elif len(state['p3_cards']) == 0:
        print(" Player 3 WINS!")
    else:
        print("Game ended after max turns. Final card counts:")
        print(f"  P1: {len(state['p1_cards'])} | P2: {len(state['p2_cards'])} | P3: {len(state['p3_cards'])}")
    print("=" * 55)
    return state


print("Main game loop ready.")

Main game loop ready.


## 9. Run Game – Simulation Mode (All AI)

In [12]:
# SIMULATION MODE
# All three players are AI. User just watches.
final_state = play_game(p3_mode='simulation')

         UNO GAME AI – Adversarial Search 
Mode: P1=Minimax(Defensive), P2=Expectimax(Offensive), P3=AI-Minimax

Starting top card: Yellow 1
P1 hand: [Yellow 6, Green Skip, Blue 6, Red 3, Green 4]
P2 hand: [Yellow 4, Green 8, Red 2, Blue 7, Green 0]
P3 hand: [Red 0, Yellow 3, Yellow Skip, Yellow 8, Red 5]
Deck size: 28 cards remaining

── Turn 1: Player 1 (Minimax) ──
   Top card : Yellow 1
   Hand     : [Yellow 6, Green Skip, Blue 6, Red 3, Green 4]
   Valid moves: [Yellow 6]
    P1 Minimax decision: Yellow 6 (score=41.0)
   Cards remaining → P1:4 | P2:5 | P3:5

── Turn 2: Player 2 (Expectimax) ──
   Top card : Yellow 6
   Hand     : [Yellow 4, Green 8, Red 2, Blue 7, Green 0]
   Valid moves: [Yellow 4]
    P2 Expectimax decision: Yellow 4 (score=47.5)
   All moves considered (depth=1 preview):
     → Yellow 4: Expected score = 47.5
     → Draw: Expected score = 39.5
   Cards remaining → P1:4 | P2:4 | P3:5

── Turn 3: Player 3 ──
   Top card : Yellow 4
   Hand     : [Red 0, Yellow 3, 

## 10. Run Game – Manual Mode (Player 3 is User)

In [13]:
# Start the game where you play as Player 3.
# You will be asked to choose your moves when it's your turn.
play_game(p3_mode='manual')

         UNO GAME AI – Adversarial Search 
Mode: P1=Minimax(Defensive), P2=Expectimax(Offensive), P3=User

Starting top card: Green 1
P1 hand: [Red 7, Green 4, Green 8, Blue 5, Red Skip]
P2 hand: [Blue 7, Yellow 1, Yellow Skip, Yellow 3, Yellow 5]
P3 hand: [Red 4, Blue 2, Yellow 0, Green 7, Yellow 2]
Deck size: 28 cards remaining

── Turn 1: Player 1 (Minimax) ──
   Top card : Green 1
   Hand     : [Red 7, Green 4, Green 8, Blue 5, Red Skip]
   Valid moves: [Green 4, Green 8]
    P1 Minimax decision: Green 4 (score=43.0)
   Cards remaining → P1:4 | P2:5 | P3:5

── Turn 2: Player 2 (Expectimax) ──
   Top card : Green 4
   Hand     : [Blue 7, Yellow 1, Yellow Skip, Yellow 3, Yellow 5]
   Valid moves: ['Draw']
    P2 Expectimax decision: Draw (score=38.64)
   All moves considered (depth=1 preview):
     → Draw: Expected score = 41.5
   Cards remaining → P1:4 | P2:6 | P3:5

── Turn 3: Player 3 ──
   Top card : Green 4
   Hand     : [Red 4, Blue 2, Yellow 0, Green 7, Yellow 2]
   Valid move

Enter index of card to play (or -1 to draw):  3


    P3 plays: Green 7
   Cards remaining → P1:4 | P2:6 | P3:4

── Turn 4: Player 1 (Minimax) ──
   Top card : Green 7
   Hand     : [Red 7, Green 8, Blue 5, Red Skip]
   Valid moves: [Red 7, Green 8]
    P1 Minimax decision: Green 8 (score=50.0)
   Cards remaining → P1:3 | P2:6 | P3:4

── Turn 5: Player 2 (Expectimax) ──
   Top card : Green 8
   Hand     : [Blue 7, Yellow 1, Yellow Skip, Yellow 3, Yellow 5, Blue 0]
   Valid moves: ['Draw']
    P2 Expectimax decision: Draw (score=37.65)
   All moves considered (depth=1 preview):
     → Draw: Expected score = 34.5
   Cards remaining → P1:3 | P2:7 | P3:4

── Turn 6: Player 3 ──
   Top card : Green 8
   Hand     : [Red 4, Blue 2, Yellow 0, Yellow 2]
   Valid moves: ['Draw']

🎴 Your hand (Player 3):
  [0] Red 4
  [1] Blue 2
  [2] Yellow 0
  [3] Yellow 2
Top card: Green 8
Valid moves: None – you must draw


Press Enter to draw a card... 


    P3 plays: Draw
   Cards remaining → P1:3 | P2:7 | P3:5

── Turn 7: Player 1 (Minimax) ──
   Top card : Green 8
   Hand     : [Red 7, Blue 5, Red Skip]
   Valid moves: ['Draw']
    P1 Minimax decision: Draw (score=45.0)
   Cards remaining → P1:4 | P2:7 | P3:5

── Turn 8: Player 2 (Expectimax) ──
   Top card : Green 8
   Hand     : [Blue 7, Yellow 1, Yellow Skip, Yellow 3, Yellow 5, Blue 0, Red 5]
   Valid moves: ['Draw']
    P2 Expectimax decision: Draw (score=34.5)
   All moves considered (depth=1 preview):
     → Draw: Expected score = 33.5
   Cards remaining → P1:4 | P2:8 | P3:5

── Turn 9: Player 3 ──
   Top card : Green 8
   Hand     : [Red 4, Blue 2, Yellow 0, Yellow 2, Green Skip]
   Valid moves: [Green Skip]

🎴 Your hand (Player 3):
  [0] Red 4
  [1] Blue 2
  [2] Yellow 0
  [3] Yellow 2
  [4] Green Skip
Top card: Green 8
Valid moves: [Green Skip]


...

Enter index of card to play (or -1 to draw):  2


    P3 plays: Yellow 6
   Cards remaining → P1:2 | P2:2 | P3:2

── Turn 43: Player 1 (Minimax) ──
   Top card : Yellow 6
   Hand     : [Red 0, Red 6]
   Valid moves: [Red 6]
    P1 Minimax decision: Red 6 (score=47.0)
   Cards remaining → P1:1 | P2:2 | P3:2

── Turn 44: Player 2 (Expectimax) ──
   Top card : Red 6
   Hand     : [Red 2, Red 3]
   Valid moves: [Red 2, Red 3]
    P2 Expectimax decision: Red 2 (score=47.5)
   All moves considered (depth=1 preview):
     → Red 2: Expected score = 50.5
     → Red 3: Expected score = 50.5
     → Draw: Expected score = 42.5
   Cards remaining → P1:1 | P2:1 | P3:2

── Turn 45: Player 3 ──
   Top card : Red 2
   Hand     : [Red 9, Blue 6]
   Valid moves: [Red 9]

🎴 Your hand (Player 3):
  [0] Red 9
  [1] Blue 6
Top card: Red 2
Valid moves: [Red 9]


Enter index of card to play (or -1 to draw):  0


    P3 plays: Red 9
   Cards remaining → P1:1 | P2:1 | P3:1

── Turn 46: Player 1 (Minimax) ──
   Top card : Red 9
   Hand     : [Red 0]
   Valid moves: [Red 0]
    P1 Minimax decision: Red 0 (score=52.0)
   Cards remaining → P1:0 | P2:1 | P3:1

  GAME OVER
 Player 1 (Minimax – Defensive) WINS!


{'p1_cards': [],
 'p2_cards': [Red 3],
 'p3_cards': [Blue 6],
 'top_card': Red 0,
 'deck': [Green 2,
  Blue 1,
  Green 6,
  Blue Skip,
  Yellow 7,
  Yellow 8,
  Red 8,
  Red 1,
  Green 9,
  Yellow 9,
  Green 3,
  Blue 9,
  Blue 4],
 'current_player': 1,
 'skip_next': False}